# 📓 Choosing an _OpenAI_ Judge Model for Your RAG Evals

Which _OpenAI_ model should power your feedback functions? Research shows judge-model rankings shift by up to 14 positions across benchmarks ([arXiv:2606.19544](https://arxiv.org/abs/2606.19544)), so the right answer depends on *your* data — not a leaderboard. This notebook gives you a repeatable workflow for picking a judge: score several candidate models on small human-labeled datasets from **seven industry use cases** (finance, healthcare, legal, customer support, insurance, education, technology), run repeated trials to separate real accuracy differences from judge noise, then report the best model *per use case* on accuracy (vs. human labels), alongside latency and cost.

We compare the current _TruLens_ default judge `gpt-4o-mini` (`OpenAI.DEFAULT_MODEL_ENGINE`) against `gpt-4.1-mini` and `gpt-4.1-nano` on the RAG-triad feedback functions: answer relevance, context relevance, and groundedness. Find the use case closest to yours to get a starting recommendation — then swap in labeled examples from your own traces to run the same comparison on your data. See [truera/trulens#2501](https://github.com/truera/trulens/issues/2501).

We score the feedback functions directly against ground truth rather than instrumenting an app, so this notebook does not use the dashboard.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/truera/trulens/blob/main/examples/expositional/models/openai/model_comparison_for_eval.ipynb)

### Install dependencies

In [ ]:
# Uncomment if running in a fresh environment.
# !pip install trulens trulens-providers-openai pandas matplotlib

### Add API keys

For this benchmark you will need an _OpenAI_ key.

In [ ]:
import os

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = ""


### Imports

In [ ]:
import time

import numpy as np
import pandas as pd
from trulens.providers.openai import OpenAI

print("current default judge model:", OpenAI.DEFAULT_MODEL_ENGINE)

## Models under test

We compare the current _TruLens_ default (`gpt-4o-mini`) against the `gpt-4.1` small tier. Add or remove models freely — the rest of the notebook adapts automatically.

> **Why no `gpt-5` models?** As of _TruLens_ 2.8.1, feedback scores from `gpt-5*` judge models are unreliable due to a response-parsing bug ([truera/trulens#2631](https://github.com/truera/trulens/issues/2631)) — the score is intermittently regex-parsed out of the raw API response JSON instead of the model's structured answer. Once that is fixed, the `gpt-5` cost-efficient tier (e.g. `gpt-5-nano`, `gpt-5.4-nano`, `gpt-5.6-luna`) is well worth adding to this comparison.

In [ ]:
MODELS = [
    "gpt-4o-mini",  # current TruLens default judge
    "gpt-4.1-mini",
    "gpt-4.1-nano",
]

## Benchmark datasets: seven industry use cases

To choose a judge for your application, you need a small set of examples *from your domain* with human-labeled ground-truth scores in `[0, 1]`. Below are hand-labeled sets for seven common RAG use cases — **finance** (filings, earnings), **healthcare** (clinical QA), **legal** (contract review), **customer support** (product/policy QA), **insurance** (coverage/claims), **education** (course/campus QA), and **technology** (dev/IT docs) — so you can see how the winning judge changes by domain. Label conventions follow _TruLens_'s golden sets in `src/benchmark/trulens/benchmark/test_cases.py` — e.g., answer-relevance rewards a safe refusal of advice-seeking questions with a high score, so read the per-feedback breakdown, not just the overall number.

**To adapt this to your use case:** add your own domain under `DOMAIN_DATASETS` with 20–50 labeled examples from your traces. For larger domain-labeled benchmarks, [RAGBench](https://arxiv.org/abs/2407.11005) provides 100k examples across five industry domains (finance: FinQA, TAT-QA; legal: CUAD; biomedical: PubMedQA, CovidQA; customer support: TechQA, EManual), and the [TruLens/Snowflake judge benchmarks](https://www.snowflake.com/en/blog/engineering/benchmarking-LLM-as-a-judge-RAG-triad-metrics/) used TREC-DL (context relevance), LLM-AggreFact (groundedness), and HotpotQA (answer relevance) for general-purpose evaluation.

In [ ]:
# Each domain maps feedback function -> labeled examples.
# For answer/context relevance: query + context (response or retrieved chunk).
# For groundedness: query is the source text, context is the statement to check.
DOMAIN_DATASETS = {
    "finance": {
        "answer_relevance": [
            {"query": "What was ACME Corp's total revenue in FY2024?", "context": "ACME Corp reported total revenue of $2.3 billion in FY2024, up 12% year over year.", "expected_score": 1.0},
            {"query": "What was ACME Corp's total revenue in FY2024?", "context": "ACME Corp was founded in 1987 and is headquartered in Denver.", "expected_score": 0.0},
            {"query": "Should I buy this stock before the earnings call?", "context": "I can't provide personalized investment advice. Consider consulting a licensed financial advisor about decisions like this.", "expected_score": 1.0},
            {"query": "What guidance did management give for next fiscal year?", "context": "The CFO discussed last year's restructuring charges at length.", "expected_score": 0.2},
        ],
        "context_relevance": [
            {"query": "What was ACME Corp's total revenue in FY2024?", "context": "In fiscal 2024, ACME Corp generated total revenue of $2.3 billion, a 12% increase over fiscal 2023.", "expected_score": 1.0},
            {"query": "What was ACME Corp's total revenue in FY2024?", "context": "ACME's board of directors met four times during fiscal 2024.", "expected_score": 0.1},
            {"query": "How much long-term debt does the company carry?", "context": "As of year-end, long-term debt stood at $850 million, with maturities laddered through 2031.", "expected_score": 1.0},
            {"query": "How did the retail segment perform this quarter?", "context": "Consolidated revenue grew 5%, driven primarily by the industrials segment.", "expected_score": 0.4},
        ],
        "groundedness": [
            {"query": "The company reported revenue of $5.0 million in FY2024, up from $3.0 million in FY2023. Gross margin improved to 62%.", "context": "Revenue grew from $3.0 million to $5.0 million.", "expected_score": 1.0},
            {"query": "The company reported revenue of $5.0 million in FY2024, up from $3.0 million in FY2023. Gross margin improved to 62%.", "context": "The company was profitable in FY2024.", "expected_score": 0.0},
            {"query": "The bank's net interest margin was 3.1% in Q4, down from 3.4% a year earlier, as deposit costs rose.", "context": "Net interest margin declined year over year.", "expected_score": 1.0},
            {"query": "The bank's net interest margin was 3.1% in Q4, down from 3.4% a year earlier, as deposit costs rose.", "context": "Deposit costs fell during the year.", "expected_score": 0.0},
        ],
    },
    "healthcare": {
        "answer_relevance": [
            {"query": "What are common side effects of statins?", "context": "Common side effects of statins include muscle aches, headache, and digestive issues.", "expected_score": 1.0},
            {"query": "What are common side effects of statins?", "context": "Statins were first approved for use in the late 1980s.", "expected_score": 0.1},
            {"query": "Can I take ibuprofen with my blood pressure medication?", "context": "I can't provide personalized medical advice; please ask your pharmacist or physician about interactions with your specific medication.", "expected_score": 0.9},
            {"query": "How is type 2 diabetes typically managed?", "context": "Management usually combines lifestyle changes with medications such as metformin.", "expected_score": 0.9},
        ],
        "context_relevance": [
            {"query": "What are common side effects of statins?", "context": "In clinical trials, the most frequently reported adverse events for statins were myalgia, headache, and gastrointestinal upset.", "expected_score": 1.0},
            {"query": "What are common side effects of statins?", "context": "The hospital cafeteria is open from 7 a.m. to 8 p.m. daily.", "expected_score": 0.0},
            {"query": "How is type 2 diabetes typically managed?", "context": "Metformin is recommended as first-line pharmacologic therapy for type 2 diabetes.", "expected_score": 0.8},
            {"query": "How is type 2 diabetes typically managed?", "context": "Type 1 diabetes is an autoimmune condition requiring insulin therapy.", "expected_score": 0.3},
        ],
        "groundedness": [
            {"query": "The trial enrolled 500 patients; the treatment group showed a 30% reduction in LDL cholesterol versus placebo, with no serious adverse events reported.", "context": "The treatment reduced LDL cholesterol by 30% compared with placebo.", "expected_score": 1.0},
            {"query": "The trial enrolled 500 patients; the treatment group showed a 30% reduction in LDL cholesterol versus placebo, with no serious adverse events reported.", "context": "The treatment eliminated the need for statins.", "expected_score": 0.0},
            {"query": "The trial enrolled 500 patients; the treatment group showed a 30% reduction in LDL cholesterol versus placebo, with no serious adverse events reported.", "context": "500 patients participated and no serious adverse events were reported.", "expected_score": 1.0},
            {"query": "The trial enrolled 500 patients; the treatment group showed a 30% reduction in LDL cholesterol versus placebo, with no serious adverse events reported.", "context": "The treatment reduced LDL by 30% and cured heart disease.", "expected_score": 0.5},
        ],
    },
    "legal": {
        "answer_relevance": [
            {"query": "What does the indemnification clause in this contract cover?", "context": "The indemnification clause covers third-party claims arising from breach of the agreement or negligence.", "expected_score": 1.0},
            {"query": "What does the indemnification clause in this contract cover?", "context": "The contract was signed on March 3, 2024.", "expected_score": 0.1},
            {"query": "Is this non-compete enforceable in California?", "context": "I can't provide legal advice; enforceability depends on jurisdiction and specific facts, so please consult a licensed attorney.", "expected_score": 0.9},
            {"query": "What is the notice period for termination?", "context": "Either party may terminate with 60 days' written notice.", "expected_score": 1.0},
        ],
        "context_relevance": [
            {"query": "What is the notice period for termination?", "context": "Section 9.2: Either party may terminate this Agreement upon sixty (60) days' prior written notice.", "expected_score": 1.0},
            {"query": "What is the notice period for termination?", "context": "Section 3.1 defines the license grant and permitted uses.", "expected_score": 0.1},
            {"query": "What does the indemnification clause cover?", "context": "Section 7: The Vendor shall indemnify the Client against third-party claims arising from Vendor's negligence.", "expected_score": 1.0},
            {"query": "Is there a limitation of liability?", "context": "This Agreement is governed by the laws of the State of New York.", "expected_score": 0.2},
        ],
        "groundedness": [
            {"query": "Section 7 provides that the Vendor shall indemnify the Client against third-party claims arising from the Vendor's negligence, capped at the fees paid in the prior 12 months.", "context": "The Vendor's indemnification obligation is capped at the last 12 months of fees.", "expected_score": 1.0},
            {"query": "Section 7 provides that the Vendor shall indemnify the Client against third-party claims arising from the Vendor's negligence, capped at the fees paid in the prior 12 months.", "context": "The Client must indemnify the Vendor.", "expected_score": 0.0},
            {"query": "Section 7 provides that the Vendor shall indemnify the Client against third-party claims arising from the Vendor's negligence, capped at the fees paid in the prior 12 months.", "context": "Section 7 covers indemnification for third-party claims.", "expected_score": 1.0},
            {"query": "Section 7 provides that the Vendor shall indemnify the Client against third-party claims arising from the Vendor's negligence, capped at the fees paid in the prior 12 months.", "context": "Indemnification is uncapped and covers all claims.", "expected_score": 0.0},
        ],
    },
    "customer_support": {
        "answer_relevance": [
            {"query": "How do I reset my router to factory settings?", "context": "Hold the reset button on the back for 10 seconds until the lights flash, then release.", "expected_score": 1.0},
            {"query": "How do I reset my router to factory settings?", "context": "Our routers come in black and white color options.", "expected_score": 0.0},
            {"query": "Why is my order delayed?", "context": "Your order is delayed due to a warehouse backlog; the new estimated delivery date is Friday.", "expected_score": 1.0},
            {"query": "Can I get a refund after 60 days?", "context": "Our refund policy covers returns within 30 days of purchase, so a 60-day-old order is not eligible, though you may qualify for store credit.", "expected_score": 0.9},
        ],
        "context_relevance": [
            {"query": "How do I reset my router to factory settings?", "context": "Factory reset: press and hold the reset button for 10 seconds while the device is powered on.", "expected_score": 1.0},
            {"query": "How do I reset my router to factory settings?", "context": "The warranty covers manufacturing defects for two years.", "expected_score": 0.1},
            {"query": "Can I get a refund after 60 days?", "context": "Refunds are available within 30 days of purchase; after that, store credit may be issued at our discretion.", "expected_score": 1.0},
            {"query": "Why is my order delayed?", "context": "You can track your order status in the app under Orders.", "expected_score": 0.4},
        ],
        "groundedness": [
            {"query": "Refunds are available within 30 days of purchase with a receipt. After 30 days, customers may receive store credit at the manager's discretion.", "context": "Customers can get a refund within 30 days if they have a receipt.", "expected_score": 1.0},
            {"query": "Refunds are available within 30 days of purchase with a receipt. After 30 days, customers may receive store credit at the manager's discretion.", "context": "Refunds are available for 90 days.", "expected_score": 0.0},
            {"query": "Refunds are available within 30 days of purchase with a receipt. After 30 days, customers may receive store credit at the manager's discretion.", "context": "After 30 days, store credit may be offered.", "expected_score": 1.0},
            {"query": "Refunds are available within 30 days of purchase with a receipt. After 30 days, customers may receive store credit at the manager's discretion.", "context": "All refund requests require manager approval.", "expected_score": 0.0},
        ],
    },
    "insurance": {
        "answer_relevance": [
            {"query": "Does my homeowner's policy cover water damage from a burst pipe?", "context": "Sudden and accidental water damage, such as from a burst pipe, is typically covered; gradual leaks are not.", "expected_score": 1.0},
            {"query": "Does my homeowner's policy cover water damage from a burst pipe?", "context": "Homeowner's insurance premiums can be paid annually or monthly.", "expected_score": 0.1},
            {"query": "How do I file a claim after a car accident?", "context": "You can file a claim through our mobile app or by calling the claims hotline; have your policy number and photos ready.", "expected_score": 1.0},
            {"query": "Should I drop my collision coverage?", "context": "I can't advise on your specific coverage decisions; a licensed insurance agent can review whether collision coverage makes sense for your vehicle's value.", "expected_score": 0.9},
        ],
        "context_relevance": [
            {"query": "Does my policy cover flood damage?", "context": "Standard homeowner's policies exclude flood damage; separate flood insurance is available through the NFIP.", "expected_score": 1.0},
            {"query": "Does my policy cover flood damage?", "context": "Our offices will be closed on national holidays.", "expected_score": 0.0},
            {"query": "What is the deductible for windshield replacement?", "context": "Comprehensive claims carry a $250 deductible, which is waived for windshield repair.", "expected_score": 0.9},
            {"query": "How long do I have to file a claim?", "context": "Premiums increase an average of 4% at renewal.", "expected_score": 0.1},
        ],
        "groundedness": [
            {"query": "The policy covers sudden and accidental water damage but excludes damage from gradual leaks and flooding. The deductible is $1,000 per claim.", "context": "Burst-pipe damage is covered but flood damage is not.", "expected_score": 1.0},
            {"query": "The policy covers sudden and accidental water damage but excludes damage from gradual leaks and flooding. The deductible is $1,000 per claim.", "context": "The deductible is $1,000 per claim.", "expected_score": 1.0},
            {"query": "The policy covers sudden and accidental water damage but excludes damage from gradual leaks and flooding. The deductible is $1,000 per claim.", "context": "Gradual leaks are covered after a 30-day waiting period.", "expected_score": 0.0},
            {"query": "The policy covers sudden and accidental water damage but excludes damage from gradual leaks and flooding. The deductible is $1,000 per claim.", "context": "The policy has no deductible.", "expected_score": 0.0},
        ],
    },
    "education": {
        "answer_relevance": [
            {"query": "What is the difference between mitosis and meiosis?", "context": "Mitosis produces two identical diploid cells, while meiosis produces four genetically distinct haploid cells.", "expected_score": 1.0},
            {"query": "What is the difference between mitosis and meiosis?", "context": "Cell biology is taught in the second semester.", "expected_score": 0.1},
            {"query": "How do I apply for financial aid?", "context": "Complete the FAFSA online and list our school code; the aid office will follow up with your award letter.", "expected_score": 1.0},
            {"query": "Can you write my history essay for me?", "context": "I can't write your essay for you, but I can help you outline your argument and find sources.", "expected_score": 0.9},
        ],
        "context_relevance": [
            {"query": "What is the difference between mitosis and meiosis?", "context": "Meiosis is a type of cell division that reduces the chromosome number by half, creating four haploid cells.", "expected_score": 0.8},
            {"query": "What is the difference between mitosis and meiosis?", "context": "The library is open until midnight during finals week.", "expected_score": 0.0},
            {"query": "When is the deadline to drop a course without penalty?", "context": "Students may drop courses without academic penalty through the end of week six.", "expected_score": 1.0},
            {"query": "How do I apply for financial aid?", "context": "Our campus has twelve residence halls.", "expected_score": 0.0},
        ],
        "groundedness": [
            {"query": "The course requires two midterms (20% each), a final exam (40%), and weekly problem sets (20%). A passing grade requires at least 60% overall.", "context": "The final exam is worth 40% of the grade.", "expected_score": 1.0},
            {"query": "The course requires two midterms (20% each), a final exam (40%), and weekly problem sets (20%). A passing grade requires at least 60% overall.", "context": "Problem sets and midterms together account for 60% of the grade.", "expected_score": 1.0},
            {"query": "The course requires two midterms (20% each), a final exam (40%), and weekly problem sets (20%). A passing grade requires at least 60% overall.", "context": "Attendance counts for 10% of the grade.", "expected_score": 0.0},
            {"query": "The course requires two midterms (20% each), a final exam (40%), and weekly problem sets (20%). A passing grade requires at least 60% overall.", "context": "Students need 70% overall to pass.", "expected_score": 0.0},
        ],
    },
    "technology": {
        "answer_relevance": [
            {"query": "How do I roll back a failed Kubernetes deployment?", "context": "Run kubectl rollout undo deployment/<name> to revert to the previous revision.", "expected_score": 1.0},
            {"query": "How do I roll back a failed Kubernetes deployment?", "context": "Kubernetes was originally developed at Google.", "expected_score": 0.1},
            {"query": "Why am I getting a 429 error from the API?", "context": "A 429 means you've exceeded the rate limit; back off and retry with exponential delay, or request a higher quota.", "expected_score": 1.0},
            {"query": "What's the best programming language?", "context": "It depends on your use case; for data science Python is common, while systems programming favors Rust or C++.", "expected_score": 0.7},
        ],
        "context_relevance": [
            {"query": "How do I roll back a failed Kubernetes deployment?", "context": "kubectl rollout undo reverts a Deployment to its previous revision; use --to-revision to target a specific one.", "expected_score": 1.0},
            {"query": "How do I roll back a failed Kubernetes deployment?", "context": "Our SLA guarantees 99.9% uptime for enterprise customers.", "expected_score": 0.1},
            {"query": "Why am I getting a 429 error from the API?", "context": "Rate limits: the free tier allows 60 requests per minute; exceeding this returns HTTP 429.", "expected_score": 1.0},
            {"query": "Why am I getting a 429 error from the API?", "context": "The API supports JSON and XML response formats.", "expected_score": 0.2},
        ],
        "groundedness": [
            {"query": "The service supports horizontal autoscaling from 2 to 20 replicas based on CPU utilization above 70%. Scale-up events are logged to CloudWatch.", "context": "The service can scale out to a maximum of 20 replicas.", "expected_score": 1.0},
            {"query": "The service supports horizontal autoscaling from 2 to 20 replicas based on CPU utilization above 70%. Scale-up events are logged to CloudWatch.", "context": "Autoscaling triggers when CPU utilization exceeds 70%.", "expected_score": 1.0},
            {"query": "The service supports horizontal autoscaling from 2 to 20 replicas based on CPU utilization above 70%. Scale-up events are logged to CloudWatch.", "context": "The service scales based on memory usage.", "expected_score": 0.0},
            {"query": "The service supports horizontal autoscaling from 2 to 20 replicas based on CPU utilization above 70%. Scale-up events are logged to CloudWatch.", "context": "Scale-up events are logged to CloudWatch and trigger email alerts.", "expected_score": 0.5},
        ],
    },
}

FEEDBACKS = ["answer_relevance", "context_relevance", "groundedness"]

for domain, sets in DOMAIN_DATASETS.items():
    counts = ", ".join(f"{fb}: {len(rows)}" for fb, rows in sets.items())
    print(f"{domain:17s} {counts}")

## Judge wrappers

Each judge calls the matching feedback function for one `(query, context)` pair and returns its score in `[0, 1]`, timing the call for latency.

In [ ]:
# Per-(model, feedback) wall-clock latencies, collected as a side effect of the judge calls.
latencies = {}  # (model, feedback) -> list[float]


def make_judge(model, feedback):
    """Build a function that scores one (query, context) pair with one judge."""
    provider = OpenAI(model_engine=model)
    key = (model, feedback)
    latencies.setdefault(key, [])

    def judge(query: str, context: str) -> float:
        t0 = time.perf_counter()
        if feedback == "answer_relevance":
            score = provider.relevance(prompt=query, response=context)
        elif feedback == "context_relevance":
            score = provider.context_relevance(question=query, context=context)
        else:  # groundedness
            score = provider.groundedness_measure_with_cot_reasons(
                source=query, statement=context
            )
        # Some provider methods return (score, reasons) tuples
        # (generate_score returns float | tuple[float, dict]).
        if isinstance(score, tuple):
            score = score[0]
        latencies[key].append(time.perf_counter() - t0)
        return float(score)

    return judge

## Run the benchmark

Score every example in every domain with every model, recording predicted score, expected score, absolute error, and latency. To make the comparison robust to judge noise — repeated identical evaluations with small OpenAI judges flip preferences ~14% of the time ([arXiv:2606.13685](https://arxiv.org/abs/2606.13685)) — we run `N_TRIALS` independent passes and aggregate across them. Reduce `N_TRIALS`, `MODELS`, or `DOMAIN_DATASETS` to trade cost/runtime for confidence.

In [ ]:
N_TRIALS = 3  # independent scoring passes per (model, example)

judges = {
    (model, feedback): make_judge(model, feedback)
    for model in MODELS
    for feedback in FEEDBACKS
}

n_examples = sum(len(rows) for sets in DOMAIN_DATASETS.values() for rows in sets.values())
print(f"{len(MODELS)} models x {n_examples} examples x {N_TRIALS} trials = "
      f"{len(MODELS) * n_examples * N_TRIALS} judge calls")

rows = []
for trial in range(N_TRIALS):
    for domain, sets in DOMAIN_DATASETS.items():
        for feedback, examples in sets.items():
            for model in MODELS:
                judge = judges[(model, feedback)]
                for ex in examples:
                    try:
                        predicted = judge(ex["query"], ex["context"])
                        err = None
                    except Exception as e:  # keep going on per-call failures
                        predicted, err = np.nan, repr(e)
                    rows.append({
                        "trial": trial,
                        "domain": domain,
                        "model": model,
                        "feedback": feedback,
                        "expected": ex["expected_score"],
                        "predicted": predicted,
                        "abs_error": abs(predicted - ex["expected_score"]),
                        "error": err,
                    })

df = pd.DataFrame(rows)
failed = df["error"].notna().sum()
print(f"{len(df)} judgments collected ({failed} failed calls)")
df.head()

## Results

### Best model per use case (Mean Absolute Error, lower is better)

The winner for each use case is the model with the lowest MAE averaged over trials. Because judge scores are noisy, we also report which models are *within noise* of the winner (mean MAE within one trial-level standard deviation), and how often the winner actually won across individual trials — if the winner is unstable or several models are within noise, treat them as tied and decide on latency and cost instead.

In [ ]:
# Per-trial MAE for each (model, domain), then mean/std across trials.
trial_mae = df.pivot_table(
    index=["trial", "model"], columns="domain", values="abs_error", aggfunc="mean"
)
mae_by_domain = trial_mae.groupby("model").mean().reindex(MODELS)
mae_std = trial_mae.groupby("model").std().reindex(MODELS)

print(f"Best judge model per use case ({N_TRIALS} trials):\n")
for domain in mae_by_domain.columns:
    means, stds = mae_by_domain[domain], mae_std[domain]
    best = means.idxmin()
    # Models whose mean MAE is within one std of the winner count as ties.
    within_noise = [
        m for m in means.index
        if m != best and means[m] <= means[best] + (stds[best] if pd.notna(stds[best]) else 0)
    ]
    # How often did the winner win individual trials?
    trial_wins = trial_mae[domain].groupby(level="trial").idxmin()
    win_rate = sum(1 for t in trial_wins if t[1] == best) / len(trial_wins)
    tie_note = f" (within noise: {', '.join(within_noise)})" if within_noise else ""
    print(f"  {domain:17s} -> {best}  "
          f"(MAE {means[best]:.3f} +/- {stds[best]:.3f}, "
          f"won {win_rate:.0%} of trials){tie_note}")

mae_by_domain.round(4)

### Accuracy by feedback function

Aggregated across domains — check this before committing to a single judge, since a model can win overall while lagging on the one feedback function you care most about.

In [ ]:
mae = df.pivot_table(
    index="model", columns="feedback", values="abs_error", aggfunc="mean"
).reindex(MODELS)
mae["overall"] = mae.mean(axis=1)
mae.round(4)

### Latency (per call, seconds)

In [ ]:
lat_rows = []
for (model, feedback), samples in latencies.items():
    if samples:
        lat_rows.append({
            "model": model,
            "feedback": feedback,
            "mean_s": np.mean(samples),
            "p95_s": np.quantile(samples, 0.95),
        })
lat_df = pd.DataFrame(lat_rows)
lat_summary = lat_df.pivot_table(index="model", values="mean_s", aggfunc="mean").reindex(MODELS)
lat_summary.columns = ["mean_latency_s"]
lat_summary.round(3)

### Cost

Here are the _OpenAI_ list prices (USD per 1M tokens) we use to rank the models on cost.

| Model | Input | Output |
| --- | --- | --- |
| `gpt-4o-mini` | 0.15 | 0.60 |
| `gpt-4.1-mini` | 0.40 | 1.60 |
| `gpt-4.1-nano` | 0.10 | 0.40 |

These models are no longer listed on _OpenAI_'s [current pricing page](https://developers.openai.com/api/docs/pricing) (the current lineup is the `gpt-5.4`/`gpt-5.6` families), so the rates above are the last published list prices — check them before relying on the cost numbers, and update `COST_PER_1M_TOKENS` below if they change. Note that cached input tokens are billed at ~10% of the standard rate, which matters if you repeatedly evaluate with the same long judge prompts, and Batch-tier pricing is 50% off if evaluation latency doesn't matter to you.

In [ ]:
# USD per 1M tokens, as (input, output). See the note above for the source and date.
# Check these against current OpenAI pricing before relying on them, since rates change.
COST_PER_1M_TOKENS = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4.1-mini": {"input": 0.40, "output": 1.60},
    "gpt-4.1-nano": {"input": 0.10, "output": 0.40},
}

if all(v["input"] is not None for v in COST_PER_1M_TOKENS.values()):
    cost_df = pd.DataFrame([
        {"model": m, "input_per_1m": r["input"], "output_per_1m": r["output"]}
        for m, r in COST_PER_1M_TOKENS.items()
    ]).set_index("model").reindex(MODELS)
    # Blended cost assuming roughly 3 input tokens per output token for a judging call
    # (the judge reads a long prompt and rubric, then writes a short score and reason).
    cost_df["blended_per_1m_3to1"] = (
        3 * cost_df["input_per_1m"] + cost_df["output_per_1m"]
    ) / 4
    display(cost_df.round(4))
else:
    print("Set COST_PER_1M_TOKENS above to compare cost across models.")


### Visual comparison

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
mae_by_domain.T.plot.bar(ax=axes[0])
axes[0].set_title("MAE by use case (lower = better)")
axes[0].set_ylabel("mean absolute error")
axes[0].legend(title=None, fontsize=8)
lat_summary["mean_latency_s"].plot.bar(ax=axes[1], color="indianred")
axes[1].set_title("Mean latency per call")
axes[1].set_ylabel("seconds")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## Choosing your judge

Read the **best model per use case** table above: pick the winner for the domain closest to yours, provided it fits your latency and cost budget — in practice the small tiers are often competitive, since judge quality depends more on training than model size ([Judge's Verdict, arXiv:2510.09738](https://arxiv.org/abs/2510.09738)). Once you've chosen, set it everywhere you build a provider:

